# 第 6 章 · 前向链与后向链

Runestone 式阅读 + CodeLens 展示规则库**每一轮**扫描与事实池变化。

> 本 notebook 采用 [Runestone Academy · PythonDS](https://runestone.academy/ns/books/published/pythonds/index.html) 式结构：
> **Reading（阅读）→ Listing（清单/伪代码）→ ActiveCode（运行）→ CodeLens（分步状态）→ Self-Check（自测）**。
> 配套交互网页见各章 `chN.html`。

## 学习目标

- 能逐步模拟前向链
- 能倒推后向链子目标
- 能判断何时用前向/后向

## 1. Reading · 推理 = 显式化隐含结论

**前向链（forward chaining）** 从已知事实出发，反复扫描规则「若条件成立则结论入池」。

**后向链（backward chaining）** 从目标出发，分解子目标直到命中事实库。

**Listing · 前向链核心循环**

```text
while 有新事实:
    for rule in rules:
        if rule.if in known and rule.then not in known:
            known.add(rule.then)
```

In [1]:
# ActiveCode · 加载规则库
import sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "labs").exists() and (ROOT.parent / "labs").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "labs"))
sys.path.insert(0, str(ROOT / "labs" / "ch06"))
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
from IPython.display import display, Markdown
from reasoning import *
from common.codelens import print_frames as print_codelens
data = load_rules()
print('事实库:', data['facts'])
print('规则:')
for r in data['rules']: print(' ', r['id'], r['if'], '⇒', r['then'])

事实库: ['人(苏格拉底)']
规则:
  R1 ['人(X)'] ⇒ 会死(X)
  R2 ['会死(X)'] ⇒ 终有一死(X)


**Self-Check** · 第一轮哪条规则会被触发？

<details><summary>点击展开答案</summary>

R1: 人(苏格拉底) ⇒ 会死(苏格拉底)

</details>

## 2. CodeLens · 前向链逐步

In [2]:
# ActiveCode · 每一轮 known 集合变化
fc = codelens_forward_chain()
print_codelens(fc)

── Step 0 ── 初始化
   执行: known = facts
   known = ['人(苏格拉底)']
   goal = '终有一死(苏格拉底)'

── Step 1 ── 第 1 轮触发 2 条规则
   执行: round 1 scan rules
   新增 = ['R1: 人(苏格拉底) ⇒ 会死(苏格拉底)', 'R2: 会死(苏格拉底) ⇒ 终有一死(苏格拉底)']
   known = ['人(苏格拉底)', '会死(苏格拉底)', '终有一死(苏格拉底)']

── Step 2 ── 检查目标是否在 known 中
   执行: check goal
   goal = '终有一死(苏格拉底)'
    proved = True



In [3]:
# ActiveCode · 文本 log 对照
for line in forward_chain(): print(line)

初始事实: 人(苏格拉底)
第 1 轮: R1: 人(苏格拉底) ⇒ 会死(苏格拉底); R2: 会死(苏格拉底) ⇒ 终有一死(苏格拉底)
目标 终有一死(苏格拉底): ✓


## 3. 后向链 CodeLens（目标驱动）

**Listing · 后向链**

```text
goal → 匹配规则 → subgoals → 直到 fact
```

In [4]:
# ActiveCode · 后向 trace
for line in backward_chain(): print(' ', line)

  目标: 终有一死(苏格拉底)
  子目标: 会死(苏格拉底) (R2)
  子目标: 人(苏格拉底) (R1)
  匹配事实: True → 证明成立


## 本章小结

前向撒网、后向聚焦；方向不同，都依赖规则完备性。